# 🎬 Imagem → vídeo com LTX-Video (Google Colab)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AntonioDantas/HelloWorld/blob/main/colab/LTX_Video_imagem_para_video.ipynb)

Transforma qualquer imagem em um vídeo curto guiado por um prompt, usando o modelo aberto
[LTX-Video 0.9.5](https://huggingface.co/Lightricks/LTX-Video-0.9.5) da Lightricks.

**Antes de começar:** *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU T4* (ou L4/A100, se tiver).

### Como usar
1. Execute a **célula 1** (prepara o ambiente e conecta o Google Drive).
2. Adicione trabalhos à fila pela **célula 2** (envia imagens do computador ou usa um arquivo do Drive)
   e/ou pela **célula 3** (várias imagens de uma vez, cada uma com seu prompt).
3. Execute a **célula 5** para gerar os vídeos. Eles ficam em `MyDrive/LTX-Video/saidas/`.

### Se a GPU acabar ou a sessão cair
Tudo fica salvo no Drive: a fila (`fila.json`), as imagens, os prompts já processados e cada vídeo assim que fica pronto.
Quando puder voltar, conecte de novo com GPU e execute **só a célula 1 e depois a célula 5**: ela continua do
próximo vídeo pendente. Só o vídeo que estava sendo gerado no momento da queda é refeito do zero.
Para parar de propósito, use o botão ⏹ da célula 5. O vídeo em andamento volta para a fila.

### Dicas de prompt
O modelo entende melhor **prompts em inglês, detalhados**, descrevendo o movimento, a câmera e a luz. Por exemplo:
> *A woman with long brown hair turns her head toward the camera and smiles softly, wind gently moving her hair,
> slow camera push-in, warm golden hour sunlight, realistic, cinematic.*

### Tempo aproximado
Na T4 gratuita, um vídeo de 4 s em 768 px com 30 passos leva alguns minutos. Na L4 ou A100, bem menos.
O primeiro uso de cada sessão também baixa o modelo (≈ 6 GB) e, se houver prompts novos, o codificador de texto (≈ 19 GB).

In [ ]:
#@title 1. Preparar ambiente  { display-mode: "form" }
#@markdown Execute sempre primeiro, inclusive ao retomar depois de uma queda.
PASTA_NO_DRIVE = "LTX-Video"  #@param {type:"string"}
PRECISAO = "auto"  #@param ["auto", "bfloat16", "float32", "float16"]
#@markdown `auto` usa bfloat16 em GPUs novas (L4, A100) e float32 na T4, que não tem bfloat16 nativo.

import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "diffusers==0.41.0", "transformers==5.20.2",
                "accelerate", "sentencepiece", "imageio", "imageio-ffmpeg"], check=True)

import base64, gc, hashlib, io, json, os, random, re, shutil, tempfile, time
from datetime import datetime

import numpy as np
import torch
from diffusers import LTXConditionPipeline
from diffusers.pipelines.ltx.pipeline_ltx_condition import LTXVideoCondition
from diffusers.utils import export_to_video
from huggingface_hub import snapshot_download
from PIL import Image, ImageOps
from transformers import T5EncoderModel

try:
    from google.colab import drive, files
    drive.mount("/content/drive")
    BASE = os.path.join("/content/drive/MyDrive", PASTA_NO_DRIVE)
    NO_COLAB = True
except ImportError:  # Jupyter fora do Colab: usa uma pasta local
    BASE = os.path.abspath(PASTA_NO_DRIVE)
    NO_COLAB = False

PASTAS = {nome: os.path.join(BASE, nome) for nome in ["entradas", "saidas", "lote", "embeddings"]}
for p in PASTAS.values():
    os.makedirs(p, exist_ok=True)
FILA = os.path.join(BASE, "fila.json")
MODELO = "Lightricks/LTX-Video-0.9.5"
NEGATIVO_PADRAO = "worst quality, inconsistent motion, blurry, jittery, distorted"
PERMITIR_CPU = False  # sem GPU levaria horas por vídeo

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
GPU_NOVA = DEVICE == "cuda" and torch.cuda.get_device_capability()[0] >= 8
if PRECISAO != "auto":
    DTYPE = getattr(torch, PRECISAO)
else:
    DTYPE = torch.bfloat16 if (GPU_NOVA or DEVICE == "cpu") else torch.float32
DTYPE_T5 = torch.bfloat16 if (GPU_NOVA or DEVICE == "cpu") else torch.float16


def agora():
    return datetime.now().strftime("%Y-%m-%d %H:%M:%S")


def ler_fila():
    if not os.path.exists(FILA):
        return []
    with open(FILA, encoding="utf-8") as f:
        return json.load(f)


def salvar_fila(fila):
    tmp = FILA + ".tmp"
    with open(tmp, "w", encoding="utf-8") as f:
        json.dump(fila, f, ensure_ascii=False, indent=1)
    os.replace(tmp, FILA)


def atualizar(job_id, **campos):
    fila = ler_fila()
    for job in fila:
        if job["id"] == job_id:
            job.update(campos, atualizado=agora())
    salvar_fila(fila)


def adicionar_trabalho(nome, dados, prompt, negativo, duracao_s, fps, lado_maior, passos, guidance, seed):
    """Copia a imagem para o Drive e coloca o trabalho na fila. Ignora duplicatas exatas."""
    prompt = prompt.strip()
    if not prompt:
        print(f"⚠️  {nome}: prompt vazio, ignorado.")
        return None
    try:
        Image.open(io.BytesIO(dados)).verify()
    except Exception:
        print(f"⚠️  {nome}: não é uma imagem válida, ignorado.")
        return None
    params = dict(prompt=prompt, negativo=negativo.strip(), duracao_s=float(duracao_s), fps=int(fps),
                  lado_maior=int(lado_maior), passos=int(passos), guidance=float(guidance))
    params["seed"] = int(seed) if int(seed) >= 0 else random.randrange(2**31)
    chave = hashlib.sha1(dados + json.dumps(params, sort_keys=True).encode()).hexdigest()[:10]
    fila = ler_fila()
    if any(job["id"] == chave for job in fila):
        print(f"↩️  {nome}: já está na fila ({chave}).")
        return chave
    nome_seguro = re.sub(r"[^\w.-]", "_", os.path.basename(nome))
    caminho = os.path.join(PASTAS["entradas"], f"{chave}_{nome_seguro}")
    with open(caminho, "wb") as f:
        f.write(dados)
    fila.append(dict(id=chave, imagem=caminho, **params, status="pendente", saida=None, erro=None, criado=agora(), atualizado=agora()))
    salvar_fila(fila)
    print(f"✅ {nome}: adicionado à fila ({chave}).")
    return chave


def preparar_imagem(caminho, lado_maior):
    """Corrige a rotação, redimensiona mantendo a proporção e ajusta para múltiplos de 32."""
    img = ImageOps.exif_transpose(Image.open(caminho)).convert("RGB")
    w0, h0 = img.size
    escala = lado_maior / max(w0, h0)
    w = max(128, round(w0 * escala / 32) * 32)
    h = max(128, round(h0 * escala / 32) * 32)
    return ImageOps.fit(img, (w, h), Image.LANCZOS)


def quadros(duracao_s, fps):
    """O LTX-Video exige 8k+1 quadros (máx. 257)."""
    return min(257, max(1, round(duracao_s * fps / 8)) * 8 + 1)


def caminho_emb(job):
    chave = hashlib.sha1(f"{MODELO}\0{job['prompt']}\0{job['negativo']}".encode()).hexdigest()[:16]
    return os.path.join(PASTAS["embeddings"], f"{chave}.pt")


def baixar(partes):
    if os.path.isdir(MODELO):
        return MODELO
    return snapshot_download(MODELO, allow_patterns=["model_index.json"] + [f"{p}/*" for p in partes])


def liberar_memoria():
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()


def codificar_prompts(jobs):
    """Roda o T5 só para prompts ainda sem cache e salva o resultado no Drive."""
    faltando = {}
    for job in jobs:
        if not os.path.exists(caminho_emb(job)):
            faltando[caminho_emb(job)] = job
    if not faltando:
        print("Prompts já codificados (cache no Drive), pulando o T5.")
        return
    print(f"Codificando {len(faltando)} prompt(s) com o T5 (o primeiro uso baixa ≈ 19 GB)...")
    raiz = baixar(["text_encoder", "tokenizer", "scheduler"])
    t5 = T5EncoderModel.from_pretrained(os.path.join(raiz, "text_encoder"), dtype=DTYPE_T5, device_map=DEVICE)
    pipe = LTXConditionPipeline.from_pretrained(raiz, text_encoder=t5, transformer=None, vae=None, torch_dtype=DTYPE_T5)
    try:
        with torch.no_grad():
            for destino, job in faltando.items():
                pe, pm, npe, npm = pipe.encode_prompt(prompt=job["prompt"], negative_prompt=job["negativo"], device=DEVICE)
                tmp = destino + ".tmp"
                torch.save({"pe": pe.cpu(), "pm": pm.cpu(), "npe": npe.cpu(), "npm": npm.cpu()}, tmp)
                os.replace(tmp, destino)
    finally:
        del pipe, t5
        liberar_memoria()


def carregar_gerador():
    print(f"Carregando o LTX-Video em {DEVICE} ({str(DTYPE).replace('torch.', '')})...")
    raiz = baixar(["transformer", "vae", "scheduler"])
    pipe = LTXConditionPipeline.from_pretrained(raiz, text_encoder=None, tokenizer=None, torch_dtype=DTYPE).to(DEVICE)
    pipe.vae.enable_tiling()  # decodifica o vídeo em blocos para caber na VRAM
    return pipe


def gerar_video(pipe, job):
    img = preparar_imagem(job["imagem"], job["lado_maior"])
    w, h = img.size
    n = quadros(job["duracao_s"], job["fps"])
    emb = torch.load(caminho_emb(job))
    print(f"▶️  {job['id']}: {w}×{h}, {n} quadros, {job['passos']} passos, seed {job['seed']}")
    frames = pipe(
        conditions=[LTXVideoCondition(image=img, frame_index=0)],
        prompt_embeds=emb["pe"].to(DEVICE, DTYPE),
        prompt_attention_mask=emb["pm"].to(DEVICE),
        negative_prompt_embeds=emb["npe"].to(DEVICE, DTYPE),
        negative_prompt_attention_mask=emb["npm"].to(DEVICE),
        width=w,
        height=h,
        num_frames=n,
        frame_rate=job["fps"],
        num_inference_steps=job["passos"],
        guidance_scale=job["guidance"],
        decode_timestep=0.05,
        decode_noise_scale=0.025,
        generator=torch.Generator("cpu").manual_seed(job["seed"]),
    ).frames[0]
    if np.asarray(frames[len(frames) // 2]).max() == 0:
        raise RuntimeError("o vídeo saiu todo preto (estouro numérico). Na célula 1, troque PRECISAO para float32.")
    nome = os.path.splitext(os.path.basename(job["imagem"]))[0]
    destino = os.path.join(PASTAS["saidas"], f"{nome}.mp4")
    with tempfile.TemporaryDirectory() as tmp:
        local = os.path.join(tmp, "video.mp4")
        export_to_video(frames, local, fps=job["fps"])
        shutil.copy(local, destino)  # grava primeiro no disco local e só depois copia para o Drive
    return destino


def mostrar_video(caminho, largura=480):
    from IPython.display import HTML, display
    with open(caminho, "rb") as f:
        dados = base64.b64encode(f.read()).decode()
    display(HTML(f'<video src="data:video/mp4;base64,{dados}" width="{largura}" controls loop autoplay muted></video>'))


def processar_fila(tempo_limite_min=0, mostrar=True):
    if DEVICE != "cuda" and not PERMITIR_CPU:
        print("❌ Sem GPU. Vá em Ambiente de execução → Alterar o tipo de ambiente de execução → GPU e rode a célula 1 de novo.")
        return
    fila = ler_fila()
    for job in fila:
        if job["status"] == "processando":  # sobrou de uma sessão que caiu
            job["status"] = "pendente"
    salvar_fila(fila)
    pendentes = [job for job in fila if job["status"] == "pendente"]
    if not pendentes:
        print("Nenhum trabalho pendente. Adicione imagens nas células 2 ou 3.")
        return
    print(f"{len(pendentes)} trabalho(s) pendente(s).")
    codificar_prompts(pendentes)
    pipe = carregar_gerador()
    inicio = time.time()
    for i, job in enumerate(pendentes, 1):
        if tempo_limite_min and time.time() - inicio > tempo_limite_min * 60:
            print(f"⏸️  Tempo limite de {tempo_limite_min} min atingido. Os {len(pendentes) - i + 1} restantes continuam na fila.")
            break
        print(f"\n[{i}/{len(pendentes)}] {os.path.basename(job['imagem'])}: {job['prompt'][:80]}")
        atualizar(job["id"], status="processando", erro=None)
        t0 = time.time()
        try:
            saida = gerar_video(pipe, job)
        except KeyboardInterrupt:
            atualizar(job["id"], status="pendente")
            print("⏹️  Interrompido. Este trabalho voltou para a fila; rode a célula 5 para continuar.")
            return
        except torch.cuda.OutOfMemoryError:
            atualizar(job["id"], status="erro", erro="memória da GPU insuficiente: diminua o lado maior ou a duração")
            print("❌ Memória da GPU insuficiente. Diminua o lado maior ou a duração e recoloque na fila (célula 4).")
            liberar_memoria()
            continue
        except Exception as e:
            atualizar(job["id"], status="erro", erro=f"{type(e).__name__}: {e}")
            print(f"❌ Erro: {type(e).__name__}: {e}")
            liberar_memoria()
            continue
        atualizar(job["id"], status="concluido", saida=saida, tempo_s=round(time.time() - t0))
        print(f"✅ Pronto em {time.time() - t0:.0f} s → {saida}")
        if mostrar:
            mostrar_video(saida)
    restantes = sum(job["status"] == "pendente" for job in ler_fila())
    print(f"\nFim. Pendentes: {restantes}. Vídeos em {PASTAS['saidas']}")


if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"✅ GPU: {props.name} ({props.total_memory / 2**30:.0f} GB), precisão {str(DTYPE).replace('torch.', '')}")
else:
    print("⚠️  Sem GPU neste ambiente. Ative uma em Ambiente de execução → Alterar o tipo de ambiente de execução.")
print(f"📁 Pasta de trabalho: {BASE}")
pend = sum(job["status"] in ("pendente", "processando") for job in ler_fila())
if pend:
    print(f"↻ Há {pend} trabalho(s) pendente(s) na fila. Para continuar, execute a célula 5.")

In [ ]:
#@title 2. Adicionar imagens à fila  { display-mode: "form" }
#@markdown Todas as imagens enviadas nesta execução usam o mesmo prompt. Para prompts diferentes, rode a célula de novo ou use a célula 3.
prompt = ""  #@param {type:"string"}
prompt_negativo = "worst quality, inconsistent motion, blurry, jittery, distorted"  #@param {type:"string"}
origem = "enviar do computador"  #@param ["enviar do computador", "arquivo no Drive"]
#@markdown Se escolheu "arquivo no Drive", informe o caminho (ex.: `/content/drive/MyDrive/fotos/praia.jpg`):
arquivo_no_drive = ""  #@param {type:"string"}
#@markdown ---
duracao_segundos = 4  #@param {type:"slider", min:1, max:10, step:0.5}
fps = 24  #@param {type:"integer"}
lado_maior_px = 768  #@param [512, 640, 768, 896, 1024] {type:"raw"}
passos = 30  #@param {type:"slider", min:8, max:50, step:1}
guidance = 3.0  #@param {type:"number"}
seed = -1  #@param {type:"integer"}
#@markdown `seed = -1` sorteia uma semente (enviar a mesma imagem de novo gera outra variação); use um número fixo para repetir um resultado.

if origem == "enviar do computador":
    if not NO_COLAB:
        raise RuntimeError("O envio pelo navegador só funciona no Colab; use 'arquivo no Drive'.")
    enviados = files.upload()
else:
    with open(arquivo_no_drive, "rb") as f:
        enviados = {os.path.basename(arquivo_no_drive): f.read()}

for nome, dados in enviados.items():
    adicionar_trabalho(nome, dados, prompt, prompt_negativo, duracao_segundos, fps, lado_maior_px, passos, guidance, seed)

In [ ]:
#@title 3. (Opcional) Adicionar várias imagens pela pasta `lote`  { display-mode: "form" }
#@markdown Coloque no Drive, em `MyDrive/LTX-Video/lote/`, cada imagem com um `.txt` de mesmo nome contendo o prompt
#@markdown (ex.: `praia.jpg` + `praia.txt`). Imagens sem `.txt` usam o prompt padrão abaixo.
#@markdown Rodar esta célula de novo não duplica o que já está na fila.
prompt_padrao = ""  #@param {type:"string"}
prompt_negativo = "worst quality, inconsistent motion, blurry, jittery, distorted"  #@param {type:"string"}
duracao_segundos = 4  #@param {type:"slider", min:1, max:10, step:0.5}
fps = 24  #@param {type:"integer"}
lado_maior_px = 768  #@param [512, 640, 768, 896, 1024] {type:"raw"}
passos = 30  #@param {type:"slider", min:8, max:50, step:1}
guidance = 3.0  #@param {type:"number"}
seed = 0  #@param {type:"integer"}
#@markdown A semente padrão aqui é fixa (0) para que rodar a célula de novo não repita trabalhos; com -1, cada execução cria novas variações.

EXTENSOES = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
imagens = sorted(n for n in os.listdir(PASTAS["lote"]) if n.lower().endswith(EXTENSOES))
if not imagens:
    print(f"Nenhuma imagem em {PASTAS['lote']}")
for nome in imagens:
    txt = os.path.join(PASTAS["lote"], os.path.splitext(nome)[0] + ".txt")
    texto = open(txt, encoding="utf-8").read() if os.path.exists(txt) else prompt_padrao
    with open(os.path.join(PASTAS["lote"], nome), "rb") as f:
        adicionar_trabalho(nome, f.read(), texto, prompt_negativo, duracao_segundos, fps, lado_maior_px, passos, guidance, seed)

In [ ]:
#@title 4. Ver e gerenciar a fila  { display-mode: "form" }
acao = "só mostrar"  #@param ["só mostrar", "recolocar os trabalhos com erro na fila", "remover os concluídos da lista", "remover os pendentes", "apagar a fila inteira"]
#@markdown Remover itens da lista não apaga os vídeos já gerados.

fila = ler_fila()
if acao == "recolocar os trabalhos com erro na fila":
    for job in fila:
        if job["status"] == "erro":
            job.update(status="pendente", erro=None)
elif acao == "remover os concluídos da lista":
    fila = [job for job in fila if job["status"] != "concluido"]
elif acao == "remover os pendentes":
    fila = [job for job in fila if job["status"] not in ("pendente", "processando")]
elif acao == "apagar a fila inteira":
    fila = []
if acao != "só mostrar":
    salvar_fila(fila)

ICONES = {"pendente": "⏳", "processando": "⚙️", "concluido": "✅", "erro": "❌"}
contagem = {s: sum(job["status"] == s for job in fila) for s in ICONES}
print("  ".join(f"{ICONES[s]} {s}: {n}" for s, n in contagem.items()))
for job in fila:
    print(f"\n{ICONES[job['status']]} [{job['id']}] {os.path.basename(job['imagem'])} · {job['duracao_s']} s · "
          f"{job['lado_maior']} px · {job['passos']} passos · seed {job['seed']}")
    print(f"   {job['prompt'][:120]}")
    if job.get("saida"):
        print(f"   → {job['saida']} ({job.get('tempo_s', '?')} s)")
    if job.get("erro"):
        print(f"   ! {job['erro']}")

In [ ]:
#@title 5. Gerar os vídeos da fila  { display-mode: "form" }
#@markdown Para retomar depois de uma queda, execute a célula 1 e depois esta.
tempo_limite_min = 0  #@param {type:"integer"}
#@markdown `tempo_limite_min`: para de começar novos vídeos depois desse tempo, para economizar a cota de GPU (0 = sem limite).
mostrar_videos = True  #@param {type:"boolean"}

processar_fila(tempo_limite_min, mostrar_videos)

In [ ]:
#@title 6. Ver ou baixar os vídeos gerados  { display-mode: "form" }
mostrar_ultimos = 6  #@param {type:"integer"}
baixar_zip = False  #@param {type:"boolean"}

videos = sorted((os.path.join(PASTAS["saidas"], n) for n in os.listdir(PASTAS["saidas"]) if n.endswith(".mp4")),
                key=os.path.getmtime, reverse=True)
print(f"{len(videos)} vídeo(s) em {PASTAS['saidas']}")
for v in videos[:mostrar_ultimos]:
    print(os.path.basename(v))
    mostrar_video(v)
if baixar_zip and videos:
    zip_path = shutil.make_archive("/content/videos_ltx", "zip", PASTAS["saidas"])
    if NO_COLAB:
        files.download(zip_path)
    else:
        print(zip_path)